<h3 style="color:#1E3A8A; font-family:Arial, Helvetica, sans-serif; margin-bottom:6px;">
  <b>IST 691 — Phase 1 (Part 2): Unique Identifier & Sentence-Level Text Segmentation</b>
</h3>

<div style="font-family:Arial, Helvetica, sans-serif; line-height:1.55;">
  <p style="margin-top:0;">
    <b>Big picture.</b> This module converts cleaned transcript-level text into sentence-aware, overlapping textual units while preserving stable identifiers from the prior stage. The resulting dataset establishes a consistent unit of analysis for belief extraction and downstream representation learning.
  </p>

  <p>
    <b>Single source of truth (input).</b> The module consumes the reduced dataset produced in Phase 1, Part 1, containing <code>hash_id</code> and cleaned transcript text. No new identifiers are created; all downstream units inherit the originating <code>hash_id</code>.
  </p>

  <p>
    <b>Core logic flow.</b>
    <ol style="margin-top:4px;">
      <li><b>Load</b> reduced transcript dataset from the local output directory.</li>
      <li><b>Tokenize</b> transcript text into sentences using a deterministic tokenizer.</li>
      <li><b>Segment</b> text into overlapping chunks guided by sentence boundaries.</li>
      <li><b>Attach metadata</b> to each chunk (<code>hash_id</code>, chunk index, chunk count, word count).</li>
      <li><b>Export</b> chunked text and summary statistics for downstream modules.</li>
    </ol>
  </p>

  <p>
    <b>Design considerations.</b> Chunk size, overlap, and minimum length thresholds are chosen to balance semantic completeness with processing efficiency. Overlap ensures continuity across segments while sentence-level boundaries reduce fragmentation artifacts.
  </p>

  <p>
    <b>Deliverables (local exports).</b>
    <ul style="margin-top:4px;">
      <li><b>Chunked LLM dataset</b>: sentence-aware text segments with inherited identifiers.</li>
      <li><b>Chunking summary</b>: size statistics, counts, and processing timestamp.</li>
    </ul>
  </p>

  <p style="margin-bottom:0;">
    <b>Console exhibit.</b> The execution prints configuration parameters, summary statistics, and example chunks to support immediate verification and traceability.
  </p>
</div>

<hr style="border:none; border-top:1px solid #ddd; margin:12px 0;">


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
====================================================================================================
IST 691 — Phase 1 (Part 2)  |  Unique Identifier & Sentence-Level Chunking
====================================================================================================

purpose
  - transform cleaned transcript-level text into sentence-aware, overlapping text units
  - preserve stable row-level traceability by propagating existing hash identifiers
  - prepare normalized, LLM-ready textual segments for downstream belief extraction

inputs (single source of truth)
  - reduced transcript dataset from Phase 1, Part 1
  - required columns: hash_id, text (cleaned transcript text)

processing (logic flow)
  1) load reduced transcript dataset produced in Phase 1, Part 1
  2) tokenize transcript text into sentences (NLTK punkt)
  3) construct overlapping text chunks using sentence boundaries
       - target chunk size (words)
       - controlled overlap to preserve semantic continuity
       - minimum chunk size threshold to filter noise
  4) attach deterministic metadata:
       - originating hash_id
       - chunk index and chunk count per transcript
       - word count per chunk
  5) export chunked dataset and summary statistics

outputs (local exports)
  - chunked transcript dataset for LLM processing
  - chunking summary report (counts, size statistics, timestamp)
  - execution log for reproducibility and inspection

console exhibit
  - processing parameters
  - summary statistics
  - example chunks with identifiers

====================================================================================================

December 2025 | Syracuse University | IST 691 Deep Learning Term Project

Dujun; Yifeng; Isha
"""


#### =============================================================================
#### 1. PACKAGE CHECKS & INSTALLATION
#### =============================================================================
import subprocess
import sys

def install_package(pkg_name):
    """Install a package via pip if it's not already installed."""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_name])
    except Exception as e:
        print(f"Error installing {pkg_name}: {e}", file=sys.stderr)

# Ensure nltk is available for sentence tokenization.
try:
    import nltk
    from nltk.tokenize import sent_tokenize
    nltk.data.find('tokenizers/punkt')
except ImportError:
    install_package("nltk")
    import nltk
    from nltk.tokenize import sent_tokenize
    nltk.download('punkt')
except LookupError:
    nltk.download('punkt')

#### =============================================================================
#### 2. SET WORKING DIRECTORY & DATA PATH (USE LOCAL OUTPUT FOLDER)
#### =============================================================================
import os
import time
import logging
from typing import List
import pandas as pd
from datetime import datetime
from tqdm import tqdm

# IMPORTANT: Set the base working directory to the OUTPUT folder.
# Since the previous stage saved its results there, we will load from that folder.
BASE_WORKING_DIR = "/Users/DJ/Dropbox/ADS_MA/IST.707/Project-707/output"

# UPDATED INPUT_FILE: use the reduced dataset output from Section 2.
INPUT_FILE = os.path.join(BASE_WORKING_DIR, "Section2_Reduced_LLM_Input_Dataset_Hash_and_Clean_Transcript_Text.csv")
# OUTPUT_DIR remains the same.
OUTPUT_DIR = BASE_WORKING_DIR

#### =============================================================================
#### 3. LOGGING SETUP
#### =============================================================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("text_chunking")
# Log file will be saved in the output folder.
log_file_path = os.path.join(OUTPUT_DIR, "section3_phase1_log.txt")
file_handler = logging.FileHandler(log_file_path)
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))
logger.addHandler(file_handler)

#### =============================================================================
#### 4. DATA LOADING
#### =============================================================================
def load_data(file_path: str) -> pd.DataFrame:
    """Load CSV data from the specified file path into a DataFrame."""
    logger.info(f"Loading data from {file_path}")
    try:
        df = pd.read_csv(file_path)
        logger.info(f"Loaded {len(df)} records from {file_path}")
        return df
    except Exception as e:
        logger.error(f"Error loading data: {e}")
        raise

#### =============================================================================
#### 5. TEXT CHUNKING
#### =============================================================================
def create_overlapping_chunks(text: str, target_size: int = 550, overlap: int = 150, min_size: int = 300) -> List[str]:
    """
    Split a text into overlapping chunks of approximately target_size words,
    with a specified overlap between consecutive chunks.
    
    Args:
        text (str): The input transcript text.
        target_size (int): Approximate target size of each chunk (in words).
        overlap (int): Number of words to overlap between chunks.
        min_size (int): Minimum number of words required for a chunk to be kept.
        
    Returns:
        List[str]: A list of text chunks.
    """
    from nltk.tokenize import sent_tokenize
    
    if not text or not isinstance(text, str):
        return []
    
    # Tokenize the text into sentences.
    sentences = sent_tokenize(text)
    
    # If the overall text is smaller than the target, return it as a single chunk.
    if len(' '.join(sentences).split()) <= target_size:
        return [text]
    
    chunks = []
    current_chunk = []
    current_word_count = 0
    
    # Process the text sentence by sentence.
    for sentence in sentences:
        sentence_words = sentence.split()
        sentence_word_count = len(sentence_words)
        
        # Check if adding this sentence exceeds target size while ensuring a minimum.
        if current_word_count + sentence_word_count > target_size and current_word_count >= min_size:
            chunks.append(' '.join(current_chunk))
            # Keep sentences for overlap: iterate backwards until desired overlap is reached.
            overlap_word_count = 0
            sentences_to_keep = []
            for s in reversed(current_chunk):
                s_word_count = len(s.split())
                if overlap_word_count + s_word_count <= overlap:
                    sentences_to_keep.insert(0, s)
                    overlap_word_count += s_word_count
                else:
                    break
            current_chunk = sentences_to_keep
            current_word_count = overlap_word_count
        
        current_chunk.append(sentence)
        current_word_count += sentence_word_count
    
    # Add the final chunk if it meets the minimum word count.
    if current_chunk and current_word_count >= min_size:
        chunks.append(' '.join(current_chunk))
    
    return chunks

def process_transcripts(df: pd.DataFrame, target_chunk_size: int = 550, overlap: int = 150, min_size: int = 300) -> pd.DataFrame:
    """
    Process a DataFrame of transcripts, creating overlapping chunks for each transcript.
    
    Args:
        df (pd.DataFrame): Input DataFrame containing 'hash_id' and 'text' columns.
        target_chunk_size (int): Target chunk size (in words).
        overlap (int): Overlap between consecutive chunks (in words).
        min_size (int): Minimum acceptable chunk size (in words).
        
    Returns:
        pd.DataFrame: DataFrame containing chunked texts and associated metadata.
    """
    logger.info(f"Chunking {len(df)} transcript entries")
    chunked_texts = []
    metadata = []
    
    # Process each transcript row.
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Chunking transcripts"):
        content = row['text']
        
        # Skip transcript if too short.
        if len(content.split()) < 50:
            continue
        
        # Generate overlapping chunks.
        chunks = create_overlapping_chunks(content, target_size=target_chunk_size, overlap=overlap, min_size=min_size)
        
        # Collect each chunk and its metadata.
        for i, chunk in enumerate(chunks):
            word_count = len(chunk.split())
            if word_count >= min_size:
                chunked_texts.append(chunk)
                metadata.append({
                    'hash_id': row['hash_id'],
                    'chunk_index': i,
                    'chunk_count': len(chunks),
                    'word_count': word_count
                })
    
    logger.info(f"Created {len(chunked_texts)} chunks for LLM processing")
    
    # Assemble the chunks and metadata into a DataFrame.
    if chunked_texts and metadata:
        chunk_df = pd.DataFrame({
            'text': chunked_texts,
            **{key: [m.get(key) for m in metadata] for key in metadata[0].keys()}
        })
    else:
        chunk_df = pd.DataFrame()
    
    return chunk_df

#### =============================================================================
#### 6. STATISTICS AND REPORTING
#### =============================================================================
def generate_summary_statistics(original_df: pd.DataFrame, chunk_df: pd.DataFrame) -> dict:
    """Generate summary statistics about the chunking process."""
    word_counts = chunk_df['word_count'].tolist() if 'word_count' in chunk_df.columns else []
    summary = {
        "original_record_count": len(original_df),
        "chunk_count": len(chunk_df),
        "avg_chunk_size": sum(word_counts) / len(word_counts) if word_counts else 0,
        "min_chunk_size": min(word_counts) if word_counts else 0,
        "max_chunk_size": max(word_counts) if word_counts else 0,
        "avg_chunks_per_transcript": chunk_df['chunk_count'].mean() if 'chunk_count' in chunk_df.columns else 0,
        "processing_timestamp": datetime.now().strftime("%Y%m%d_%H%M%S")
    }
    return summary

def print_summary(summary: dict) -> None:
    """Print a formatted summary of the chunking process."""
    print("\n" + "="*80)
    print("TEXT CHUNKING SUMMARY")
    print("="*80)
    print(f"Original records: {summary['original_record_count']}")
    print(f"Created chunks: {summary['chunk_count']}")
    print(f"Average chunk size: {summary['avg_chunk_size']:.1f} words")
    print(f"Chunk size range: {summary['min_chunk_size']} to {summary['max_chunk_size']} words")
    print(f"Average chunks per transcript: {summary['avg_chunks_per_transcript']:.1f}")
    print("="*80 + "\n")

#### =============================================================================
#### 7. MAIN PIPELINE FUNCTION
#### =============================================================================
def run_chunking_pipeline(
    input_file: str = INPUT_FILE,
    output_dir: str = OUTPUT_DIR,
    target_chunk_size: int = 550,
    overlap: int = 150,
    min_size: int = 300
) -> dict:
    """
    Run the chunking pipeline to prepare transcripts for LLM processing.
    
    Args:
        input_file (str): Path to the input CSV file.
        output_dir (str): Directory where output files will be saved.
        target_chunk_size (int): Target chunk size in words.
        overlap (int): Overlap between chunks in words.
        min_size (int): Minimum chunk size in words to keep.
        
    Returns:
        dict: A dictionary containing the original DataFrame, chunked DataFrame, and summary statistics.
    """
    start_time = time.time()
    logger.info("Starting Section 3 Phase 1: Text Chunking for LLM Processing (Local, Simple)")
    
    try:
        # Load input data from the output folder.
        df = load_data(input_file)
        
        # Since our input data already has two columns ('hash_id' and 'text'),
        # we do not need to rename any column.
        
        # Generate text chunks.
        chunk_df = process_transcripts(
            df, 
            target_chunk_size=target_chunk_size,
            overlap=overlap,
            min_size=min_size
        )
        
        # Save chunked transcripts.
        chunk_output_path = os.path.join(output_dir, "Section3_Phase1_Chunked_Transcripts_For_LLM.csv")
        chunk_df.to_csv(chunk_output_path, index=False)
        logger.info(f"Chunked transcripts saved to: {chunk_output_path}")
        
        # Generate and save summary statistics.
        summary = generate_summary_statistics(df, chunk_df)
        summary_path = os.path.join(output_dir, "Section3_Phase1_Chunking_Summary.txt")
        with open(summary_path, "w", encoding='utf-8') as f:
            for key, value in summary.items():
                f.write(f"{key}: {value}\n")
        
        print_summary(summary)
        
        # Show a few example chunks.
        print("Example chunks:")
        for i, (_, row) in enumerate(chunk_df.head(3).iterrows()):
            print(f"\nChunk {i+1} (hash_id: {row['hash_id']}, {row['word_count']} words):")
            print(f"{row['text'][:150]}...")
        
        elapsed_time = time.time() - start_time
        logger.info(f"Pipeline completed in {elapsed_time:.2f} seconds")
        
        return {
            'original_df': df,
            'chunk_df': chunk_df,
            'summary': summary
        }
    except Exception as e:
        logger.error(f"Error in pipeline: {str(e)}", exc_info=True)
        raise

#### =============================================================================
#### 8. MAIN DRIVER FUNCTION (NOTE: Adjusted for Notebook Style)
#### =============================================================================
def main():
    """Main driver function for local text chunking (Notebook style)."""
    import argparse

    # Setup argument parsing to ignore extra kernel arguments in a notebook.
    parser = argparse.ArgumentParser(
        description="Section 3 Phase 1: Text Chunking for LLM Processing (Local, Simple)"
    )
    parser.add_argument("--input", default=INPUT_FILE, help="Input file path (default: reduced LLM dataset in output folder)")
    parser.add_argument("--output-dir", default=OUTPUT_DIR, help="Output directory (default: output folder)")
    parser.add_argument("--chunk-size", type=int, default=550, help="Target chunk size in words")
    parser.add_argument("--overlap", type=int, default=150, help="Overlap between chunks in words")
    parser.add_argument("--min-size", type=int, default=300, help="Minimum chunk size to keep")
    
    # Use parse_known_args to ignore extra arguments from the Jupyter kernel.
    args, unknown = parser.parse_known_args()
    
    print("\n" + "="*80)
    print("SECTION 3 - PHASE 1: TEXT CHUNKING FOR LLM PROCESSING (LOCAL, SIMPLE)")
    print("="*80)
    print("Parameters:")
    print(f"  - Chunk Size: {args.chunk_size} words")
    print(f"  - Chunk Overlap: {args.overlap} words")
    print(f"  - Minimum Chunk Size: {args.min_size} words")
    print(f"  - Input File: {args.input}")
    print(f"  - Output Directory: {args.output_dir}")
    print("="*80 + "\n")
    
    run_chunking_pipeline(
        input_file=args.input,
        output_dir=args.output_dir,
        target_chunk_size=args.chunk_size,
        overlap=args.overlap,
        min_size=args.min_size
    )
    
    print("\nChunking pipeline completed successfully.")

# For notebook usage or direct execution, call main().
if __name__ == "__main__":
    main()



SECTION 3 - PHASE 1: TEXT CHUNKING FOR LLM PROCESSING (LOCAL, SIMPLE)
Parameters:
  - Chunk Size: 550 words
  - Chunk Overlap: 150 words
  - Minimum Chunk Size: 300 words
  - Input File: /Users/DJ/Dropbox/ADS_MA/IST.707/Project-707/output/Section2_Reduced_LLM_Input_Dataset_Hash_and_Clean_Transcript_Text.csv
  - Output Directory: /Users/DJ/Dropbox/ADS_MA/IST.707/Project-707/output

2025-05-06 19:04:54,442 - text_chunking - INFO - Starting Section 3 Phase 1: Text Chunking for LLM Processing (Local, Simple)
2025-05-06 19:04:54,443 - text_chunking - INFO - Loading data from /Users/DJ/Dropbox/ADS_MA/IST.707/Project-707/output/Section2_Reduced_LLM_Input_Dataset_Hash_and_Clean_Transcript_Text.csv
2025-05-06 19:04:56,982 - text_chunking - INFO - Loaded 368613 records from /Users/DJ/Dropbox/ADS_MA/IST.707/Project-707/output/Section2_Reduced_LLM_Input_Dataset_Hash_and_Clean_Transcript_Text.csv
2025-05-06 19:04:56,982 - text_chunking - INFO - Chunking 368613 transcript entries


Chunking transcripts: 100%|██████████| 368613/368613 [00:42<00:00, 8576.69it/s] 

2025-05-06 19:05:39,986 - text_chunking - INFO - Created 140236 chunks for LLM processing


2025-05-06 19:05:42,941 - text_chunking - INFO - Chunked transcripts saved to: /Users/DJ/Dropbox/ADS_MA/IST.707/Project-707/output/Section3_Phase1_Chunked_Transcripts_For_LLM.csv

TEXT CHUNKING SUMMARY
Original records: 368613
Created chunks: 140236
Average chunk size: 474.7 words
Chunk size range: 300 to 550 words
Average chunks per transcript: 4.1

Example chunks:

Chunk 1 (hash_id: 200e3c38a7cf3d17, 466 words):
Thank you, Jasmine. Happy new year, everyone, and good morning. Welcome to Ford's December 2014 sales call. I'm joined today by John Felice, our Vice ...

Chunk 2 (hash_id: fc8f8b4298e3f56c, 543 words):
Well, thank you, Erich, and good morning, everyone, and welcome to 2015. Taking a look at results for the month, Ford Motor Company sales totaled 220,...

Chunk 3 (hash_id: fc8f8b4298e3f56c, 466 words):
Meanwhile, Fusion sales of 23,166 vehicles were down a bit in December due primarily to lower fleet buyings, which were off 30%. For the year, Fusion ...
2025-05-06 19:05:42,94